# Imports

In [1]:
import pathlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

# Pegando os dados

In [2]:
df_escalonado = pd.read_csv("data/base_escalonada.csv")
TARGET = "Close"
DATE_COLUMN = "Date"

In [3]:
# pensando em 3 dias para prever
h = int(len(df_escalonado) * 0.25)
train = df_escalonado[TARGET].iloc[:-h]

test  = df_escalonado[TARGET].iloc[-h:]

# Buscando o melhor XGBoost

In [4]:
import optuna
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

# Cria variáveis defasadas para evitar vazamento temporal
dados = df_escalonado.copy()

colunas_features = [
    coluna for coluna in dados.columns
    if coluna not in [TARGET, DATE_COLUMN]
]

X = dados[colunas_features].shift(1)
X[f"{TARGET}_lag_1"] = dados[TARGET].shift(1)
X[f"{TARGET}_lag_2"] = dados[TARGET].shift(2)
X[f"{TARGET}_lag_3"] = dados[TARGET].shift(3)

y = dados[TARGET]

base_modelo = pd.concat([X, y], axis=1).dropna()
X = base_modelo.drop(columns=[TARGET])
y = base_modelo[TARGET]

h_modelo = int(len(X) * 0.25)
X_train = X.iloc[:-h_modelo]
X_test = X.iloc[-h_modelo:]
y_train = y.iloc[:-h_modelo]
y_test = y.iloc[-h_modelo:]

tscv = TimeSeriesSplit(n_splits=5)

def objetivo(trial):
    modelo = XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 100, 800),
        max_depth=trial.suggest_int("max_depth", 3, 15),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        objective="reg:squarederror",
        eval_metric="mae",
        random_state=42,
        n_jobs=-1
    )

    erros = []

    for treino_idx, validacao_idx in tscv.split(X_train):
        modelo.fit(
            X_train.iloc[treino_idx],
            y_train.iloc[treino_idx]
        )

        previsoes = modelo.predict(X_train.iloc[validacao_idx])
        erros.append(
            mean_absolute_error(
                y_train.iloc[validacao_idx],
                previsoes
            )
        )

    return np.mean(erros)

estudo = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

estudo.optimize(objetivo, n_trials=50, show_progress_bar=True)

melhor_modelo = XGBRegressor(
    **estudo.best_params,
    objective="reg:squarederror",
    eval_metric="mae",
    random_state=42,
    n_jobs=-1
)

melhor_modelo.fit(X_train, y_train)

previsoes_teste = melhor_modelo.predict(X_test)
mae_teste = mean_absolute_error(y_test, previsoes_teste)

print("Melhores parâmetros:")
print(estudo.best_params)
print(f"MAE médio na validação: {estudo.best_value:.6f}")
print(f"MAE no teste: {mae_teste:.6f}")

[I 2026-09-29 10:24:50,256] A new study created in memory with name: no-name-cfaafe37-8193-492b-b1d6-cddce6460e74
Best trial: 0. Best value: 0.419907:   2%|▏         | 1/50 [00:11<09:04, 11.10s/it]

[I 2026-09-29 10:25:01,360] Trial 0 finished with value: 0.4199073272690125 and parameters: {'n_estimators': 362, 'max_depth': 15, 'learning_rate': 0.1205712628744377, 'min_child_weight': 6, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893}. Best is trial 0 with value: 0.4199073272690125.


Best trial: 1. Best value: 0.324575:   4%|▍         | 2/50 [00:25<10:14, 12.80s/it]

[I 2026-09-29 10:25:15,344] Trial 1 finished with value: 0.32457474064563907 and parameters: {'n_estimators': 521, 'max_depth': 12, 'learning_rate': 0.010725209743171996, 'min_child_weight': 10, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'reg_alpha': 4.329370014459266e-07, 'reg_lambda': 4.4734294104626844e-07}. Best is trial 1 with value: 0.32457474064563907.


Best trial: 1. Best value: 0.324575:   6%|▌         | 3/50 [00:32<08:09, 10.42s/it]

...

[I 2026-09-29 10:28:56,630] Trial 47 finished with value: 0.2999719188490982 and parameters: {'n_estimators': 119, 'max_depth': 4, 'learning_rate': 0.021486776282693878, 'min_child_weight': 2, 'subsample': 0.8007549304288856, 'colsample_bytree': 0.8082878002066394, 'reg_alpha': 1.2001405737853703, 'reg_lambda': 0.010581464922251501}. Best is trial 46 with value: 0.2980705574789375.


Best trial: 46. Best value: 0.298071:  98%|█████████▊| 49/50 [04:09<00:03,  3.05s/it]

[I 2026-09-29 10:28:59,856] Trial 48 finished with value: 0.299787157838 and parameters: {'n_estimators': 204, 'max_depth': 4, 'learning_rate': 0.019404386910756945, 'min_child_weight': 3, 'subsample': 0.7120661928946935, 'colsample_bytree': 0.8649071433762203, 'reg_alpha': 3.615400501211066, 'reg_lambda': 9.9375256720738e-06}. Best is trial 46 with value: 0.2980705574789375.


Best trial: 46. Best value: 0.298071: 100%|██████████| 50/50 [04:13<00:00,  5.06s/it]


[I 2026-09-29 10:29:03,468] Trial 49 finished with value: 0.2981833977413023 and parameters: {'n_estimators': 234, 'max_depth': 3, 'learning_rate': 0.01244338072340272, 'min_child_weight': 2, 'subsample': 0.775398384893919, 'colsample_bytree': 0.8122653899217398, 'reg_alpha': 6.362206767437523, 'reg_lambda': 6.588253051497494e-05}. Best is trial 46 with value: 0.2980705574789375.
Melhores parâmetros:
{'n_estimators': 123, 'max_depth': 4, 'learning_rate': 0.012584867416378816, 'min_child_weight': 2, 'subsample': 0.7896090148517696, 'colsample_bytree': 0.8298466534914405, 'reg_alpha': 4.837732117198189, 'reg_lambda': 0.0037907684439583613}
MAE médio na validação: 0.298071
MAE no teste: 0.494778


In [5]:
melhores_params = estudo.best_params
MAE_FINAL = mae_teste

In [6]:
print("Melhores params:", melhores_params)
print("MAE final:", MAE_FINAL)

Melhores params: {'n_estimators': 123, 'max_depth': 4, 'learning_rate': 0.012584867416378816, 'min_child_weight': 2, 'subsample': 0.7896090148517696, 'colsample_bytree': 0.8298466534914405, 'reg_alpha': 4.837732117198189, 'reg_lambda': 0.0037907684439583613}
MAE final: 0.4947780632464398
